# 1 — Single Product URL Resolution

Use this notebook for one product when you need the complete, clean decision package:

```text
input
→ product interpretation
→ manufacturer-first search
→ retailer/country fallback
→ global fallback
→ browser + text + image validation
→ strict URL acceptance
→ business judgment sequence
→ final URL and artifacts
```

The first result view is business-facing. Engineering diagnostics follow underneath.


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "docker-compose.yml").is_file() and (candidate / "src" / "product_evidence_harness").is_dir():
            return candidate
    raise RuntimeError("Could not locate the web_search_tool repository root")

PROJECT_ROOT = find_project_root()
for required in (str(PROJECT_ROOT), str(PROJECT_ROOT / "src")):
    if required not in sys.path:
        sys.path.insert(0, required)
importlib.invalidate_caches()

from product_evidence_harness.notebook_runtime import (
    DEFAULT_FEATURE_SET,
    ensure_platform_ready,
    host_artifact_dir,
    run_product,
)
from product_evidence_harness.artifact_diagnostics import (
    build_artifact_diagnostics,
    plot_business_judgement_timeline,
)

health, platform_recovery = ensure_platform_ready(
    PROJECT_ROOT,
    auto_recover=True,
    clean_build=True,
)
platform_readiness_df = pd.DataFrame([{
    "repository": str(PROJECT_ROOT),
    "platform_status": health.get("status"),
    "runtime_contract": health.get("runtime_contract_version"),
    "manufacturer_first_primary_url": health.get("manufacturer_first_primary_url"),
    "business_judgement_review_artifact": health.get("business_judgement_review_artifact"),
    "auto_recovery_attempted": platform_recovery.attempted,
    "auto_recovery_succeeded": platform_recovery.recovered,
    "recovery_elapsed_seconds": platform_recovery.elapsed_seconds,
}])
display(platform_readiness_df)


## Product input

`main_text` and `country_code` are mandatory. Keep EAN/GTIN as text so leading zeroes are preserved. Use a new `row_id` for every execution.


In [ ]:
FEATURE_SET = DEFAULT_FEATURE_SET
RUN_SINGLE_PRODUCT = False

product = {
    "row_id": "ROW-001",
    "main_text": "Replace with the product main text",
    "country_code": "CH",
    "retailer_name": None,
    "ean": None,
    "language_code": None,
}

display(pd.DataFrame([product]))


## Execute

Set `RUN_SINGLE_PRODUCT = True` only after replacing the sample input.


In [ ]:
if RUN_SINGLE_PRODUCT:
    if product["main_text"].startswith("Replace with"):
        raise ValueError("Replace product['main_text'] before running")
    result = run_product(product, FEATURE_SET)
    artifact_path = host_artifact_dir(PROJECT_ROOT, result)
    if artifact_path is None or not artifact_path.is_dir():
        raise RuntimeError(f"Artifact directory was not created: {artifact_path}")
    diagnostics = build_artifact_diagnostics(artifact_path)
    print(f"Completed: {result.get('job_status')} | artifact={artifact_path}")
else:
    print("Ready. Replace the product input and set RUN_SINGLE_PRODUCT = True.")


# Business decision view

This is the primary review section. It shows the final result and the observable sequence:

```text
evidence → rule → judgment → next action
```

It does not expose hidden chain-of-thought.


In [ ]:
if "diagnostics" not in globals():
    raise RuntimeError("Run the product execution cell first")

source_selection = result.get("source_selection") or {}
url_delivery = result.get("url_delivery") or {}
primary_url_acceptance = result.get("primary_url_acceptance") or {}
final_decision_df = pd.DataFrame([{
    "job_status": result.get("job_status"),
    "primary_url": result.get("primary_url"),
    "primary_url_role": result.get("primary_url_role"),
    "manufacturer_url": result.get("manufacturer_url"),
    "retailer_url": result.get("retailer_url"),
    "source_selection_policy": source_selection.get("policy"),
    "source_selection_reason": source_selection.get("selection_reason"),
    "strict_primary_accepted": primary_url_acceptance.get("accepted"),
    "url_delivered": url_delivery.get("delivered"),
    "strictly_verified": url_delivery.get("strictly_verified"),
}])
display(final_decision_df)
display(diagnostics.product_input_df)

judgement_columns = [
    column for column in (
        "sequence_number",
        "decision_stage",
        "business_question",
        "evidence_considered",
        "agent_judgement",
        "judgement_status",
        "business_rule_applied",
        "effect_on_next_action",
        "visual_evidence_used",
        "visual_evidence_details",
        "alternatives_considered",
        "alternative_rejected",
        "rejection_reason",
        "confidence",
        "final_outcome",
    )
    if column in diagnostics.business_judgement_steps_df
]
display(diagnostics.business_judgement_steps_df[judgement_columns])
display(diagnostics.visual_evidence_summary_df)

review_path = artifact_path / "business_judgement_review.md"
print(f"SHARE WITH HUMAN CODER: {review_path}")


In [ ]:
fig = plot_business_judgement_timeline(diagnostics, max_steps=20)
plt.show()


# Supporting engineering diagnostics

Use these tables after reviewing the business judgment sequence.


In [ ]:
display(diagnostics.search_steps_df)

candidate_columns = [
    column for column in (
        "url",
        "source_role",
        "source_tier_name",
        "validation_status",
        "identity_status",
        "identity_accepted",
        "coverage",
        "strict_selected",
        "review_selected",
        "decision_reasons",
        "rejection_reasons",
    )
    if column in diagnostics.candidates_df
]
display(diagnostics.candidates_df[candidate_columns or list(diagnostics.candidates_df.columns)])

feature_columns = [
    column for column in (
        "url",
        "source_role",
        "feature_id",
        "feature_name",
        "value",
        "status",
        "confidence",
        "extraction_method",
        "evidence_location",
        "evidence_text",
    )
    if column in diagnostics.feature_evidence_df
]
display(diagnostics.feature_evidence_df[feature_columns or list(diagnostics.feature_evidence_df.columns)])
display(diagnostics.artifact_inventory_df)


## Export the review workbook


In [ ]:
workbook_path = artifact_path / "single_product_diagnostics.xlsx"
with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
    for sheet_name, frame in {
        "final_decision": final_decision_df,
        "overview": diagnostics.overview_df,
        "product_input": diagnostics.product_input_df,
        "business_judgments": diagnostics.business_judgement_steps_df,
        "visual_evidence_impact": diagnostics.visual_evidence_summary_df,
        "search_route": diagnostics.search_steps_df,
        "candidates": diagnostics.candidates_df,
        "feature_evidence": diagnostics.feature_evidence_df,
        "evidence_ledger": diagnostics.evidence_ledger_df,
        "belief_updates": diagnostics.belief_updates_df,
        "artifact_inventory": diagnostics.artifact_inventory_df,
    }.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print(f"Workbook: {workbook_path}")
print(f"Artifact directory: {artifact_path}")
